<a href="https://colab.research.google.com/github/EseoheB/climate-emission-strategy-analysis/blob/main/notebooks/climate_trace_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Stage 0: Environment Setup and Data Extraction

Mount Google Drive, extract the dataset, and confirm the CSV and
documentation PDF are present.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
file_path = '/content/drive/MyDrive/climate-trace-project/ers_plan_global.zip'

import zipfile
with zipfile.ZipFile(file_path, 'r') as zip_ref:
    zip_ref.extractall('/content/climate_trace_data')

import os
for root, dirs, files in os.walk('/content/climate_trace_data'):
    for f in files:
        print(os.path.join(root, f))

/content/climate_trace_data/ers_plan_global/.DS_Store
/content/climate_trace_data/ers_plan_global/DATA/ers_plan_global_v5_9_0.csv
/content/climate_trace_data/ers_plan_global/ABOUT_THE_DATA/about_the_data_v5_9_0.pdf


## Stage 1: First Look at the Data

Load the dataset and validate its structure before analysis.

In [10]:
import pandas as pd

csv_path = '/content/climate_trace_data/ers_plan_global/DATA/ers_plan_global_v5_9_0.csv'
df = pd.read_csv(csv_path)

### 1.1 Data Integrity Check

Confirm row/column count and correct data types.

In [11]:
print(df.shape)
df.dtypes

(2406480, 10)


,0
source_id,float64
source_name,object
iso3_country,object
original_inventory_sector,object
strategy_id,int64
strategy_name,object
total_emissions_reduced_per_year,float64
gas,object
strategy_description,object
difficulty_score,float64


### 1.2 Row Preview

Inspect sample rows to understand the data structure.

In [12]:
df.head()

,source_id,source_name,iso3_country,original_inventory_sector,strategy_id,strategy_name,total_emissions_reduced_per_year,gas,strategy_description,difficulty_score
0,46010669.0,Xingfeng,CHN,solid-waste-disposal,87268,Diversion and gas capture (sanitary landfill),2.452151e+06,co2e_100yr,Divert waste from sanitary landfills through r...,1.000000
1,46010667.0,West New Territories Landfill,HKG,solid-waste-disposal,87265,Diversion and gas capture (sanitary landfill),2.256181e+06,co2e_100yr,Divert waste from sanitary landfills through r...,1.000001
2,46010381.0,South East New Territories Landfill,HKG,solid-waste-disposal,86975,Diversion and gas capture (sanitary landfill),1.688938e+06,co2e_100yr,Divert waste from sanitary landfills through r...,1.000005
3,46009838.0,North East Territories Landfill,HKG,solid-waste-disposal,86420,Diversion and gas capture (sanitary landfill),9.518370e+05,co2e_100yr,Divert waste from sanitary landfills through r...,1.000023
4,46015391.0,OpenStreetMap Landfill,BRA,solid-waste-disposal,112138,Diversion and improve gas capture,5.499513e+05,co2e_100yr,Divert waste from sanitary landfills through r...,1.000051


### 1.3 Category Counts

Count unique sectors, strategies, countries, and gas types.

In [13]:
print("Sectors:", df['original_inventory_sector'].nunique())
print("Strategies:", df['strategy_name'].nunique())
print("Countries:", df['iso3_country'].nunique())
print("Gas types:", df['gas'].unique())

Sectors: 63
Strategies: 86
Countries: 249
Gas types: ['co2e_100yr']


### 1.4 Missing Data Check

Counting nulls per column before any analysis.

In [14]:
df.isnull().sum()

,0
source_id,1236478
source_name,0
iso3_country,0
original_inventory_sector,0
strategy_id,0
strategy_name,0
total_emissions_reduced_per_year,0
gas,0
strategy_description,43813
difficulty_score,0


## Stage 2: Sector-Level Concentration

Group by sector to see where reduction potential is concentrated across
the 63 sectors, before looking at individual strategies.

In [15]:
sector_totals = df.groupby('original_inventory_sector')['total_emissions_reduced_per_year'].sum()
sector_totals = sector_totals.sort_values(ascending=False)
sector_totals.head(15)

,total_emissions_reduced_per_year
original_inventory_sector,
electricity-generation,8.927630e+09
road-transportation,2.932534e+09
forest-land-clearing,2.596473e+09
forest-land-fires,2.208304e+09
iron-and-steel,1.985533e+09
residential-onsite-fuel-usage,1.601300e+09
cement,1.404207e+09
cropland-fires,1.390414e+09
coal-mining,9.163595e+08


### 2.1 Concentration Check

Calculate what share of global reduction potential the top 15 sectors
represent.

In [16]:
top_15_total = sector_totals.head(15).sum()
global_total = df['total_emissions_reduced_per_year'].sum()
concentration_pct = (top_15_total / global_total) * 100

print(f"Global total: {global_total:,.0f} tonnes CO2e/year")
print(f"Top 15 sectors: {top_15_total:,.0f} tonnes CO2e/year")
print(f"Concentration: {concentration_pct:.1f}%")

Global total: 29,941,326,464 tonnes CO2e/year
Top 15 sectors: 26,296,552,685 tonnes CO2e/year
Concentration: 87.8%


## Stage 3: Sector x Strategy Ranking

Group by sector and strategy together to see every option available
within each sector, not just the sector totals.

In [17]:
strategy_summary = df.groupby(['original_inventory_sector', 'strategy_name']).agg(
    total_reduction=('total_emissions_reduced_per_year', 'sum'),
    n_sources=('source_id', 'count'),
    avg_difficulty=('difficulty_score', 'mean')
).reset_index()

strategy_summary.shape

(157, 5)

### 3.1 Priority Score

Add a difficulty-adjusted score to compare raw impact against ease of
implementation.

In [18]:
strategy_summary['priority_score'] = strategy_summary['total_reduction'] / strategy_summary['avg_difficulty']
strategy_summary.head()

,original_inventory_sector,strategy_name,total_reduction,n_sources,avg_difficulty,priority_score
0,aluminum,Benchmark-based retrofits,1.913450e+07,62,5.072822,3.771964e+06
1,aluminum,Recycled feedstock for smelting,1.915341e+08,163,3.488472,5.490487e+07
2,aluminum,Unspecified solution,5.904427e+06,26,6.029717,9.792212e+05
3,aluminum,shut down refineries,2.960023e+06,16,7.208155,4.106492e+05
4,bauxite-mining,Electrification of mine equipment,4.760646e+06,185,7.286516,6.533501e+05


### 3.2 Ranking Within Each Sector

Rank strategies by raw impact and by priority score, within each sector,
to find where the two rankings disagree.

In [19]:
strategy_summary['rank_by_impact'] = strategy_summary.groupby('original_inventory_sector')['total_reduction'] \
    .rank(ascending=False, method='min').astype(int)

strategy_summary['rank_by_priority'] = strategy_summary.groupby('original_inventory_sector')['priority_score'] \
    .rank(ascending=False, method='min').astype(int)

strategy_summary = strategy_summary.sort_values(['original_inventory_sector', 'rank_by_impact'])
strategy_summary.head(10)

,original_inventory_sector,strategy_name,total_reduction,n_sources,avg_difficulty,priority_score,rank_by_impact,rank_by_priority
1,aluminum,Recycled feedstock for smelting,1.915341e+08,163,3.488472,5.490487e+07,1,1
0,aluminum,Benchmark-based retrofits,1.913450e+07,62,5.072822,3.771964e+06,2,2
2,aluminum,Unspecified solution,5.904427e+06,26,6.029717,9.792212e+05,3,3
3,aluminum,shut down refineries,2.960023e+06,16,7.208155,4.106492e+05,4,4
4,bauxite-mining,Electrification of mine equipment,4.760646e+06,185,7.286516,6.533501e+05,1,1
5,bauxite-mining,Unspecified solution,1.099494e+05,1,9.999620,1.099535e+04,2,2
6,biological-treatment-of-solid-waste-and-biogenic,CH4 Flaring,8.023443e+06,0,10.000000,8.023443e+05,1,1
9,cement,Carbon capture and storage,1.124167e+09,1206,6.584476,1.707299e+08,1,1
10,cement,Clinker substitution,1.392282e+08,640,5.806170,2.397935e+07,2,2
11,cement,Unspecified solution,7.382713e+07,1,9.999947,7.382752e+06,3,3


### 3.3 Finding Where the Rankings Disagree

In [20]:
disagreement = strategy_summary[strategy_summary['rank_by_impact'] != strategy_summary['rank_by_priority']]
disagreement[['original_inventory_sector', 'strategy_name', 'total_reduction',
              'avg_difficulty', 'rank_by_impact', 'rank_by_priority']]

,original_inventory_sector,strategy_name,total_reduction,avg_difficulty,rank_by_impact,rank_by_priority
35,domestic-wastewater-treatment-and-discharge,Primary Treatment to Secondary Treatment,7.597221e+05,5.630970,5,6
32,domestic-wastewater-treatment-and-discharge,Cover lagoon,5.142839e+05,3.629155,6,5
45,enteric-fermentation-cattle-operation,Unspecified solution,6.092144e+07,10.000000,1,2
44,enteric-fermentation-cattle-operation,Feed additives in cattle diet,2.944937e+07,3.834042,2,1
98,oil-and-gas-refining,Unspecified solution,4.275361e+06,9.589723,5,6
91,oil-and-gas-refining,Benchmark-based retrofits,3.181562e+06,6.015662,6,5
92,oil-and-gas-refining,Convert to biorefinery,1.262647e+06,5.667938,7,8
94,oil-and-gas-refining,Do not reopen,8.221163e+05,3.391770,8,7
142,solid-waste-disposal,Unspecified solution,4.721554e+07,9.981494,1,5
134,solid-waste-disposal,Diversion and biocover (sanitary landfill),4.043446e+07,3.018877,2,1


### 3.4 Worked Example: Solid Waste Disposal

Isolating the sector with the largest ranking disagreement to show the
impact-vs-priority tradeoff concretely.

In [21]:
swd = strategy_summary[strategy_summary['original_inventory_sector'] == 'solid-waste-disposal']

print("Ranked by raw impact:")
display(swd.sort_values('rank_by_impact')[['strategy_name', 'total_reduction', 'avg_difficulty', 'rank_by_impact']])

print("\nRanked by priority score:")
display(swd.sort_values('rank_by_priority')[['strategy_name', 'total_reduction', 'avg_difficulty', 'rank_by_priority']])

Ranked by raw impact:


,strategy_name,total_reduction,avg_difficulty,rank_by_impact
142,Unspecified solution,4.721554e+07,9.981494,1
134,Diversion and biocover (sanitary landfill),4.043446e+07,3.018877,2
139,Diversion and improve gas capture,3.575519e+07,3.466364,3
132,Diversion and Management,2.393641e+07,4.311061,4
136,Diversion and daily/intermediate cover (sanita...,1.424624e+07,4.552786,5
138,Diversion and gas capture (sanitary landfill),1.369312e+07,1.649068,6
140,Diversion and shut down,1.043161e+07,6.696076,7
131,Benchmark-based retrofits,5.283600e+06,4.172325,8
133,Diversion and biocover (controlled dumpsite),7.809421e+05,4.099785,9
137,Diversion and gas capture (controlled dumpsite),4.505489e+05,3.507037,10



Ranked by priority score:


,strategy_name,total_reduction,avg_difficulty,rank_by_priority
134,Diversion and biocover (sanitary landfill),4.043446e+07,3.018877,1
139,Diversion and improve gas capture,3.575519e+07,3.466364,2
138,Diversion and gas capture (sanitary landfill),1.369312e+07,1.649068,3
132,Diversion and Management,2.393641e+07,4.311061,4
142,Unspecified solution,4.721554e+07,9.981494,5
136,Diversion and daily/intermediate cover (sanita...,1.424624e+07,4.552786,6
140,Diversion and shut down,1.043161e+07,6.696076,7
131,Benchmark-based retrofits,5.283600e+06,4.172325,8
133,Diversion and biocover (controlled dumpsite),7.809421e+05,4.099785,9
137,Diversion and gas capture (controlled dumpsite),4.505489e+05,3.507037,10
